# Gradient Boosting

Gradient boosting juga melatih model secara berurutan, tetapi setiap model baru dilatih pada sisa kesalahan model sebelumnya. Berbeda dengan AdaBoost yang mengubah bobot sampel, gradient boosting memakai gradien fungsi kerugian.

## Tujuan

- Memahami hubungan learning rate dan jumlah estimator.
- Membandingkan kurva akurasi latih dan uji.
- Menyetel hyperparameter dengan grid search.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn import metrics
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import GridSearchCV, train_test_split


In [ ]:
def get_accuracy(X_train, X_test, y_train, y_test, model):
    return {
        "test Accuracy": metrics.accuracy_score(y_test, model.predict(X_test)),
        "train Accuracy": metrics.accuracy_score(y_train, model.predict(X_train)),
    }


## Data churn pelanggan


In [ ]:
CHURN_URL = (
    "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/"
    "IBMDeveloperSkillsNetwork-ML0101EN-SkillsNetwork/labs/Module%203/data/"
    "ChurnData.csv"
)
churn_data = pd.read_csv(CHURN_URL)
churn_data = churn_data[
    ["tenure", "age", "address", "income", "ed", "employ", "equip",
     "callcard", "wireless", "churn"]
]
churn_data["churn"] = churn_data["churn"].astype("int")

features = churn_data[["tenure", "age", "address", "income", "ed", "employ", "equip"]]
target = churn_data["churn"]

X_train, X_test, y_train, y_test = train_test_split(
    features, target, test_size=0.3, random_state=0
)
print("Train set", X_train.shape, y_train.shape)
print("Test set", X_test.shape, y_test.shape)


## Kurva akurasi terhadap jumlah estimator

Learning rate kecil butuh lebih banyak pohon. Learning rate besar mencapai akurasi tinggi lebih cepat, tetapi kurva uji bisa menurun karena overfitting.


In [ ]:
learning_rates = [0.2, 0.4, 0.6, 1.0]
estimator_counts = [10, 25, 50, 100]

train_curve = {}
test_curve = {}

for rate in learning_rates:
    train_values, test_values = [], []
    for count in estimator_counts:
        model = GradientBoostingClassifier(
            n_estimators=count, learning_rate=rate, random_state=0
        ).fit(X_train, y_train)
        accuracy = get_accuracy(X_train, X_test, y_train, y_test, model)
        train_values.append(accuracy["train Accuracy"])
        test_values.append(accuracy["test Accuracy"])
    train_curve[rate] = train_values
    test_curve[rate] = test_values

pd.DataFrame(test_curve, index=estimator_counts)


In [ ]:
figure, axis = plt.subplots(figsize=(10, 6))
for rate in learning_rates:
    axis.plot(estimator_counts, train_curve[rate], linestyle="--",
              label=f"train lr={rate}")
    axis.plot(estimator_counts, test_curve[rate], linestyle="-",
              label=f"test lr={rate}")
axis.set(xlabel="Number of estimators", ylabel="Accuracy")
axis.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()


## Grid search

Pencarian mencakup jumlah estimator, learning rate, subsample, dan kedalaman pohon.


In [ ]:
search = GridSearchCV(
    GradientBoostingClassifier(random_state=0),
    param_grid={
        "n_estimators": [50, 100],
        "learning_rate": [0.1, 0.5],
        "subsample": [0.5, 1.0],
        "max_depth": [1, 3],
    },
    scoring="accuracy",
    cv=3,
    n_jobs=-1,
).fit(X_train, y_train)
search.best_score_


In [ ]:
search.best_params_


In [ ]:
print(get_accuracy(X_train, X_test, y_train, y_test, search.best_estimator_))


## Data sel tumor


In [ ]:
CANCER_URL = (
    "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/"
    "IBMDeveloperSkillsNetwork-ML0101EN-SkillsNetwork/labs/Module%203/data/"
    "cell_samples.csv"
)
cell_data = pd.read_csv(CANCER_URL)
cell_data = cell_data[pd.to_numeric(cell_data["BareNuc"], errors="coerce").notnull()]

cell_features = cell_data[
    ["Clump", "UnifSize", "UnifShape", "MargAdh", "SingEpiSize",
     "BareNuc", "BlandChrom", "NormNucl", "Mit"]
].astype("int")
cell_target = cell_data["Class"].astype("int")

X_train, X_test, y_train, y_test = train_test_split(
    cell_features, cell_target, test_size=0.2, random_state=4
)


In [ ]:
cancer_search = GridSearchCV(
    GradientBoostingClassifier(random_state=0),
    param_grid={"n_estimators": [50, 100], "learning_rate": [0.1, 0.5],
                "max_depth": [1, 3]},
    scoring="accuracy",
    cv=3,
    n_jobs=-1,
).fit(X_train, y_train)

cancer_search.best_score_, cancer_search.best_params_


In [ ]:
print(get_accuracy(X_train, X_test, y_train, y_test, cancer_search.best_estimator_))


## Ringkasan

Gradient boosting memberi akurasi tinggi dengan jumlah pohon yang relatif sedikit. Learning rate dan jumlah estimator harus disetel bersama: menurunkan learning rate menuntut lebih banyak pohon agar hasilnya setara.
